# 19.11 下載一份權重，就能從中斷處繼續訓練嗎？


直接回答與接續訓練，需要不同的包裹。回答要有完整神經權重、模型結構、字表和預處理規則；回到中斷的那一步，還需要 optimizer 的更新紀錄、隨機狀態、取樣位置與訓練進度。只拿到回答器的權重，不能還原這些狀態。

本章有4份公開推論輸出，固定在[Hugging Face 這一版](https://huggingface.co/birdhackor/tiny-perceptron-course-models/tree/979cdfacc588ad0536f1c64fff96f264571cf054/selftrained/v2)：模型庫為 `birdhackor/tiny-perceptron-course-models`，revision 為 `979cdfacc588ad0536f1c64fff96f264571cf054`。revision固定一份不可變的發布版本，讓後來的下載仍指向同一批檔案。

| 版本名稱 | 模型庫內的目錄 | 用途 |
| --- | --- | --- |
| `moe-pretrain` | `selftrained/v2/moe-pretrain` | 觀察文字預訓練階段的推論權重。 |
| `moe-sft` | `selftrained/v2/moe-sft` | 觀察加入對話示範後的推論權重。 |
| `moe-joint` | `selftrained/v2/moe-joint` | 本章選定的有限多模態 MoE 成品。 |
| `dense-joint` | `selftrained/v2/dense-joint` | 從自己的隨機起點訓練的 Dense 對照成品。 |

每份都包含4個檔案：`model.safetensors`保存完整神經權重，`model-config.json`描述結構，`tokenizer.json`保存字元與ID對應，`inference-manifest.json`記錄來源、預處理版本與檔案內容指紋。下載程式核對指紋，載入器再核對結構與所有權重；公開檔案不含 optimizer checkpoint。前兩份是階段比較用的權重，不具有 joint 版的完整訓練歷程與能力。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
# @title 本節圖解
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/p6-19-delivery-package-roles.svg")))

圖中上路取得的是可試用的回答器。下路才有自己的完整訓練狀態：`best.pt`是驗證選出的起點，`latest.pt`是最近一次保存的進度。兩者即使都含權重，也不是同一用途。

<details>
<summary>操作：在CPU下載公開MoE並回答一個文字問題</summary>

沿用[19.1](https://birdhackor.github.io/tiny-perceptron-vlm/19.1.html)的專案環境，所有指令都在 repository 根目錄執行。若尚未裝好CPU依賴，先執行：

```bash
uv sync --frozen --extra cpu --extra selftrained
```

接著執行已核對的文字示例。`chat.py`會匿名下載上述4個檔案，存入 `outputs/selftrained-v2/public/moe-joint`，核對後直接在CPU生成；不需要私人 `.pt` 或登入。

```bash
uv run --extra cpu --extra selftrained python scripts/selftrained/chat.py \
  --model-dir outputs/selftrained-v2/public/moe-joint \
  --asset-dir outputs/selftrained-v2/data \
  --repo birdhackor/tiny-perceptron-course-models \
  --revision 979cdfacc588ad0536f1c64fff96f264571cf054 \
  --prefix selftrained/v2/moe-joint \
  --manifest-sha256 f27856892b0c46ca81ba43bfadf7051ed6bd31d848e4868a05147e5799df1e5e \
  --messages docs/selftrained/examples/v2/text.messages.json \
  --task text --device cpu --max-new-tokens 128 --threads 2
```

訊息檔先要求兩點格式，再問「App錯誤訊息一直重複出現。」本次CPU示例的 `answer` 是：

> 1. 先檢查網路並重新啟動App。  
> 2. 仍失敗再詢問官方客服。

JSON中的 `generations` 保留實際生成，`model`標示選定權重。相同下載可重用；若目的地已有不同內容，程式會拒絕覆寫，請改用新的 `--model-dir`。文字示例不讀圖音資產；想試圖片、OCR、錄音與真正後續對話，按[完整CPU操作頁](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/selftrained/v2-public-cpu-commands.md)先取得固定素材，再執行對應命令。該頁的7組情境用來檢查安裝與介面，都是已知的驗證示例，不是未知問題的成功率。

</details>

自己從零訓練則使用[本機訓練指引](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/selftrained/TRAINING.md)的 stage wrapper。`--init-checkpoint`接自己前段已完成、驗證選出的 `best.pt`，開啟新階段與新 optimizer；`--resume`接自己同階段的 `latest.pt`，恢復 optimizer、隨機與取樣狀態，繼續到原本的總步數。wrapper只接自己建立、可信任的工作檔，核對保存的資料、設定與執行記錄，而且每次 attempt 用新目錄保存輸出。這條路已支援本機執行，不依賴作者的私人訓練儲存空間。

練習判斷兩個需求：「前一階段完成後，開啟下一階段」應選自己的 `best.pt`與 `--init-checkpoint`；「聯合訓練中斷後接回原進度」應選自己的 `latest.pt`與 `--resume`。公開 safetensors 只供推論與評測，目前 trainer 沒有把它當成訓練起點的載入器，所以不能將上方下載路徑直接代入這兩項。
